# Data Lake Medallion Architecture with remote-store

Build a **Bronze / Silver / Gold** data lake using `remote-store` as the
storage layer -- entirely in-memory, no cloud credentials needed.

**Scenario**: A small manufacturing floor has temperature and vibration sensors
on 8 machines. Raw readings arrive every ten minutes with real-world data quality
issues: nulls, duplicates, out-of-range spikes, and inconsistent formats.
We'll ingest, clean, aggregate, and query this data through the medallion layers.

**What you'll learn**:
1. Use `Store.child()` to isolate Bronze / Silver / Gold namespaces
2. Write and read Parquet via the PyArrow filesystem adapter
3. Transform data between layers (clean, deduplicate, partition, aggregate)
4. Query Gold-layer analytics with DuckDB

**Dependencies**: `remote-store[arrow]`, `polars`, `duckdb`

## 1. Set Up the Data Lake

We create an in-memory store and carve out three child stores -- one per
medallion layer. In production, swap `MemoryBackend` for S3 or Azure
with zero code changes.

In [ ]:
from remote_store import Store
from remote_store.backends import MemoryBackend

lake = Store(backend=MemoryBackend())

bronze = lake.child("bronze")
silver = lake.child("silver")
gold = lake.child("gold")

print("Lake layers ready:")
print(f"  bronze -> {bronze}")
print(f"  silver -> {silver}")
print(f"  gold   -> {gold}")

## 2. Generate Raw Sensor Data

Simulate 3 days of readings from 8 sensors across a factory floor.
The data has intentional quality problems -- just like production:

- **Nulls**: ~5% of readings are missing
- **Duplicates**: ~3% of rows are exact duplicates (re-transmitted)
- **Out-of-range**: occasional temperature spikes above physical limits
- **Type drift**: some sensor IDs arrive as integers instead of strings

In [ ]:
import random
from datetime import datetime, timedelta

random.seed(42)

SENSORS = ["sensor-01", "sensor-02", "sensor-03", "sensor-04", "sensor-05", "sensor-06", "sensor-07", "sensor-08"]
MACHINES = ["press-A", "press-B", "lathe-1", "lathe-2", "welder-1", "welder-2", "oven-1", "oven-2"]
START = datetime(2026, 3, 1)
DAYS = 3
READINGS_PER_HOUR = 6  # every 10 minutes

raw_rows = []
for day_offset in range(DAYS):
    for hour in range(24):
        for reading in range(READINGS_PER_HOUR):
            for i, sensor_id in enumerate(SENSORS):
                ts = START + timedelta(days=day_offset, hours=hour, minutes=reading * 10)

                # Normal temperature range: 18-85C depending on machine
                base_temp = 35 + i * 6
                temp = round(base_temp + random.gauss(0, 3), 1)
                vibration = round(random.uniform(0.1, 5.0), 2)

                # Inject quality issues
                # ~5% null readings
                if random.random() < 0.05:
                    temp = None
                # ~2% out-of-range spikes
                elif random.random() < 0.02:
                    temp = round(random.uniform(200, 500), 1)
                # ~2% type drift: sensor ID as integer
                sid = i + 1 if random.random() < 0.02 else sensor_id

                raw_rows.append(
                    {
                        "timestamp": ts.isoformat(),
                        "sensor_id": str(sid),
                        "machine": MACHINES[i],
                        "temperature_c": temp,
                        "vibration_mm_s": vibration,
                    }
                )

# Add ~3% duplicate rows (simulating re-transmission)
n_dupes = int(len(raw_rows) * 0.03)
dupes = random.choices(raw_rows, k=n_dupes)
raw_rows.extend(dupes)
random.shuffle(raw_rows)

print(f"Generated {len(raw_rows)} raw rows")
print(f"  Date range: {START.date()} to {(START + timedelta(days=DAYS - 1)).date()}")
print(f"  Sensors: {len(SENSORS)}")
print(f"  Includes ~{n_dupes} duplicates, ~5% nulls, ~2% out-of-range")

## 3. Bronze Layer -- Raw Ingestion

Bronze stores data exactly as received. No cleaning, no transformations.
We write one Parquet file per day using the PyArrow filesystem adapter.

In [ ]:
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

from remote_store.ext.arrow import pyarrow_fs

bronze_fs = pyarrow_fs(bronze)

# Build a PyArrow table from raw rows
raw_table = pa.table(
    {
        "timestamp": [r["timestamp"] for r in raw_rows],
        "sensor_id": [r["sensor_id"] for r in raw_rows],
        "machine": [r["machine"] for r in raw_rows],
        "temperature_c": [r["temperature_c"] for r in raw_rows],
        "vibration_mm_s": [r["vibration_mm_s"] for r in raw_rows],
    }
)

# Write one file per day
for day_offset in range(DAYS):
    date_str = (START + timedelta(days=day_offset)).strftime("%Y-%m-%d")
    # Filter rows for this day (string prefix match on ISO timestamp)
    mask = pc.starts_with(raw_table["timestamp"], date_str)
    day_table = raw_table.filter(mask)
    path = f"readings/{date_str}.parquet"
    pq.write_table(day_table, path, filesystem=bronze_fs)
    print(f"  Bronze: {path} ({day_table.num_rows} rows)")

# Verify what's in bronze
print("\nBronze files:")
for f in bronze.list_files("readings", recursive=True):
    print(f"  {f.path} ({f.size:,} bytes)")

## 4. Silver Layer -- Clean, Validate, Deduplicate

Silver is where data becomes trustworthy. We apply these transformations:

1. **Parse timestamps** to proper datetime
2. **Normalize sensor IDs** (fix type-drifted integer IDs)
3. **Remove duplicates** (exact row deduplication)
4. **Drop nulls** in critical columns
5. **Filter out-of-range** temperature values (physical limit: 0-150C)
6. **Partition by date** using PyArrow's Hive-style partitioning

In [ ]:
import polars as pl

# Read all bronze files back through the PyArrow filesystem
bronze_dataset = pq.read_table("readings", filesystem=bronze_fs)
df = pl.from_arrow(bronze_dataset)
print(f"Bronze loaded: {len(df)} rows")

# --- Cleaning pipeline ---
silver_df = (
    df
    # 1. Parse timestamps
    .with_columns(
        pl.col("timestamp").str.to_datetime().alias("timestamp"),
    )
    # 2. Normalize sensor IDs: "1" -> "sensor-01", "2" -> "sensor-02", etc.
    .with_columns(
        pl.when(pl.col("sensor_id").str.len_chars() <= 2)
        .then(pl.col("sensor_id").str.zfill(2).str.replace(r"^", "sensor-"))
        .otherwise(pl.col("sensor_id"))
        .alias("sensor_id"),
    )
    # 3. Remove exact duplicates
    .unique()
    # 4. Drop rows with null temperature (critical measurement)
    .filter(pl.col("temperature_c").is_not_null())
    # 5. Filter out-of-range temperatures (keep 0-150C)
    .filter(pl.col("temperature_c").is_between(0, 150))
    # Sort for deterministic output
    .sort("timestamp", "sensor_id")
)

removed = len(df) - len(silver_df)
print(f"Silver cleaned: {len(silver_df)} rows ({removed} removed)")
print(f"  Duplicates + nulls + out-of-range = {removed} rows dropped")
print("\nSample (first 3 rows):")
for row in silver_df.head(3).iter_rows(named=True):
    print(
        f"  {row['timestamp']}  {row['sensor_id']}  {row['machine']}"
        f"  temp={row['temperature_c']}C  vib={row['vibration_mm_s']}mm/s"
    )

In [ ]:
# Write silver as Hive-partitioned Parquet (partitioned by date)
import pyarrow.dataset as ds

silver_fs = pyarrow_fs(silver)

# Add a date column for partitioning
silver_with_date = silver_df.with_columns(
    pl.col("timestamp").dt.date().cast(pl.String).alias("date"),
)

ds.write_dataset(
    silver_with_date.to_arrow(),
    "readings",
    filesystem=silver_fs,
    format="parquet",
    partitioning=ds.partitioning(pa.schema([("date", pa.string())])),
    existing_data_behavior="overwrite_or_ignore",
)

print("Silver partitions written:")
for f in silver.list_files("readings", recursive=True):
    print(f"  {f.path} ({f.size:,} bytes)")

## 5. Gold Layer -- Business-Ready Aggregates

Gold contains pre-computed metrics for dashboards and alerts.
We build two gold tables:

1. **Hourly machine stats**: avg/min/max temperature and vibration per machine per hour
2. **Daily sensor health**: reading count, temperature variability, alert count per sensor per day

In [ ]:
gold_fs = pyarrow_fs(gold)

# --- Gold table 1: Hourly machine stats ---
hourly_stats = (
    silver_df.with_columns(
        pl.col("timestamp").dt.truncate("1h").alias("hour"),
    )
    .group_by("hour", "machine", "sensor_id")
    .agg(
        pl.col("temperature_c").mean().round(1).alias("avg_temp_c"),
        pl.col("temperature_c").min().alias("min_temp_c"),
        pl.col("temperature_c").max().alias("max_temp_c"),
        pl.col("vibration_mm_s").mean().round(2).alias("avg_vibration"),
        pl.col("vibration_mm_s").max().alias("max_vibration"),
        pl.len().alias("reading_count"),
    )
    # Flag hours where max temp exceeded warning threshold (75C)
    .with_columns(
        (pl.col("max_temp_c") > 75).alias("temp_alert"),
        (pl.col("max_vibration") > 4.5).alias("vibration_alert"),
    )
    .sort("hour", "machine")
)

pq.write_table(
    hourly_stats.to_arrow(),
    "hourly_machine_stats.parquet",
    filesystem=gold_fs,
)
print(f"Gold: hourly_machine_stats.parquet ({len(hourly_stats)} rows)")
alerts = hourly_stats.filter(pl.col("temp_alert"))
print(f"  Temperature alert hours: {len(alerts)}")

In [ ]:
# --- Gold table 2: Daily sensor health report ---
daily_health = (
    silver_df.with_columns(
        pl.col("timestamp").dt.date().alias("date"),
    )
    .group_by("date", "sensor_id", "machine")
    .agg(
        pl.len().alias("total_readings"),
        pl.col("temperature_c").mean().round(1).alias("avg_temp_c"),
        pl.col("temperature_c").std().round(2).alias("temp_std_dev"),
        (pl.col("temperature_c") > 75).sum().alias("temp_alerts"),
        (pl.col("vibration_mm_s") > 4.5).sum().alias("vibration_alerts"),
    )
    .sort("date", "sensor_id")
)

pq.write_table(
    daily_health.to_arrow(),
    "daily_sensor_health.parquet",
    filesystem=gold_fs,
)
print(f"Gold: daily_sensor_health.parquet ({len(daily_health)} rows)")
print(f"  Covers {daily_health['date'].n_unique()} days x {daily_health['sensor_id'].n_unique()} sensors")

## 6. Query Gold with DuckDB

DuckDB reads PyArrow tables directly -- no import step, no copies.
These queries demonstrate the kind of analytics you'd run on gold data.

In [ ]:
import duckdb

# Load gold tables back through the PyArrow filesystem
hourly = pq.read_table("hourly_machine_stats.parquet", filesystem=gold_fs)
health = pq.read_table("daily_sensor_health.parquet", filesystem=gold_fs)

# Query 1: Which machines had the most temperature alerts?
print("=== Top Machines by Temperature Alerts ===")
result = duckdb.sql("""
    SELECT machine,
           sensor_id,
           COUNT(*) FILTER (WHERE temp_alert) as alert_hours,
           ROUND(AVG(avg_temp_c), 1) as overall_avg_temp
    FROM hourly
    GROUP BY machine, sensor_id
    HAVING COUNT(*) FILTER (WHERE temp_alert) > 0
    ORDER BY alert_hours DESC
    LIMIT 5
""")
print(f"  {'machine':<12} {'sensor':<12} {'alert_hrs':>10} {'avg_temp':>10}")
print(f"  {'-' * 12} {'-' * 12} {'-' * 10} {'-' * 10}")
for row in result.fetchall():
    print(f"  {row[0]:<12} {row[1]:<12} {row[2]:>10} {row[3]:>10}")
print()

In [ ]:
# Query 2: Hourly temperature trend for the hottest machine (first day)
print("=== Hourly Temperature Trend (Hottest Machine, Mar 1) ===")
result = duckdb.sql("""
    WITH hottest AS (
        SELECT machine, sensor_id
        FROM hourly
        GROUP BY machine, sensor_id
        ORDER BY AVG(avg_temp_c) DESC
        LIMIT 1
    )
    SELECT h.hour,
           h.avg_temp_c,
           h.max_temp_c,
           CASE WHEN h.temp_alert THEN '!!!' ELSE '' END as alert
    FROM hourly h
    JOIN hottest t ON h.machine = t.machine AND h.sensor_id = t.sensor_id
    WHERE h.hour >= '2026-03-01' AND h.hour < '2026-03-02'
    ORDER BY h.hour
""")
rows = result.fetchall()
print(f"  {'hour':<22} {'avg_temp':>9} {'max_temp':>9} {'alert':>6}")
print(f"  {'-' * 22} {'-' * 9} {'-' * 9} {'-' * 6}")
for row in rows:
    print(f"  {str(row[0]):<22} {row[1]:>9} {row[2]:>9} {row[3]:>6}")
print()

In [ ]:
# Query 3: Daily sensor health dashboard
print("=== Daily Sensor Health Dashboard ===")
result = duckdb.sql("""
    SELECT date,
           COUNT(*) as sensors,
           SUM(total_readings) as total_readings,
           ROUND(AVG(avg_temp_c), 1) as plant_avg_temp,
           SUM(temp_alerts) as temp_alerts,
           SUM(vibration_alerts) as vib_alerts
    FROM health
    GROUP BY date
    ORDER BY date
""")
print(f"  {'date':<12} {'sensors':>8} {'readings':>9} {'avg_temp':>9} {'t_alerts':>9} {'v_alerts':>9}")
print(f"  {'-' * 12} {'-' * 8} {'-' * 9} {'-' * 9} {'-' * 9} {'-' * 9}")
for row in result.fetchall():
    print(f"  {row[0]!s:<12} {row[1]:>8} {row[2]:>9} {row[3]:>9} {row[4]:>9} {row[5]:>9}")
print()

In [ ]:
# Query 4: Sensors with high temperature variability (potential issues)
print("=== Sensors with High Temperature Variability ===")
result = duckdb.sql("""
    SELECT sensor_id,
           machine,
           ROUND(AVG(avg_temp_c), 1) as mean_temp,
           ROUND(AVG(temp_std_dev), 2) as avg_std_dev,
           SUM(temp_alerts) as total_alerts
    FROM health
    GROUP BY sensor_id, machine
    ORDER BY avg_std_dev DESC
""")
print(f"  {'sensor':<12} {'machine':<12} {'mean_temp':>10} {'std_dev':>8} {'alerts':>7}")
print(f"  {'-' * 12} {'-' * 12} {'-' * 10} {'-' * 8} {'-' * 7}")
for row in result.fetchall():
    print(f"  {row[0]:<12} {row[1]:<12} {row[2]:>10} {row[3]:>8} {row[4]:>7}")

## 7. Inspect the Full Lake

Let's see the complete file layout across all three layers.

In [ ]:
total_bytes = 0
for layer_name, layer_store in [("bronze", bronze), ("silver", silver), ("gold", gold)]:
    files = list(layer_store.list_files("", recursive=True))
    layer_bytes = sum(f.size for f in files)
    total_bytes += layer_bytes
    print(f"{layer_name}/ ({len(files)} files, {layer_bytes:,} bytes)")
    for f in files:
        print(f"  {f.path}")

print(f"\nTotal lake size: {total_bytes:,} bytes")

## 8. Adapting This to Your Situation

This notebook ran entirely on `MemoryBackend`. To deploy to production,
change only the backend config:

```python
# Development (this notebook)
lake = Store(backend=MemoryBackend())

# Production -- S3
config = RegistryConfig(
    backends={"lake": BackendConfig(type="s3", options={"bucket": "my-data-lake"})},
    stores={"lake": StoreProfile(backend="lake", root_path="v1")},
)

# Production -- Azure Data Lake Storage
config = RegistryConfig(
    backends={"lake": BackendConfig(type="azure", options={
        "account_name": "myaccount",
        "container": "datalake",
    })},
    stores={"lake": StoreProfile(backend="lake", root_path="v1")},
)
```

Everything else -- `child()` layers, PyArrow writes, Polars transforms,
DuckDB queries -- stays identical.

### Key patterns to reuse

| Pattern | Code | When to use |
|---------|------|-------------|
| Layer isolation | `lake.child("bronze")` | Separate namespaces without separate credentials |
| PyArrow bridge | `pyarrow_fs(store)` | Any tool that accepts a PyArrow filesystem |
| Hive partitioning | `ds.write_dataset(..., partitioning=...)` | Time-series or categorical data |
| DuckDB on PyArrow | `duckdb.sql("SELECT ... FROM arrow_table")` | Ad-hoc analytics without ETL |
| Config switching | `RegistryConfig(...)` | Dev/staging/prod without code changes |

For more details, see the
[Data Lake Patterns guide](../../docs/how-to/data-lake-patterns.md).

## Cleanup

Nothing to clean up -- `MemoryBackend` is garbage-collected automatically.

In [ ]:
lake.close()
print("Done. The entire pipeline ran in-memory -- no files on disk, no credentials needed.")